# NeuroSLM — Bio-Inspired Language Model (~258M)

Neuroscience-inspired LM with genome-encoded algorithms, differential attention, mixture-of-depths, NT-modulated attention, predictive coding, Hebbian traces, and epigenetic self-optimization.

**Training resumes automatically across runtime disconnects.**
Checkpoints + memory + evolved DNA are persisted to Git LFS.

| Preset | Params | GPU | VRAM | Batch |
|--------|--------|-----|------|-------|
| `large` | ~100M | T4 | 15GB | 4 |
| **`xl`** | **~258M** | **A100** | **40GB** | **4** |

**Novel mechanisms (no existing model has all of these):**
1. Differential attention (dual-softmax noise cancellation)
2. Mixture-of-Depths (dynamic per-token layer skipping)
3. NT-modulated attention temperature (per-head, state-dependent)
4. Predictive coding inter-layer loss (deep supervision)
5. Hebbian fast-weight attention traces (in-context learning accelerator)
6. Genome-encoded module algorithms (evolvable via epigenetic optimizer)

**Setup:** Runtime → Change runtime type → **A100** → Set `GITHUB_TOKEN` in cell 2

In [ ]:
# 1) Accelerator + torch check
import subprocess, sys

print(f'Python {sys.version}')

# Check CUDA
r = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(torch.__version__); print(torch.cuda.is_available())"],
    capture_output=True, text=True
)
lines = r.stdout.strip().split('\n')
torch_ver = lines[0] if lines else '?'
has_cuda  = len(lines) > 1 and lines[1] == 'True'
print(f'torch {torch_ver}')

# Check TPU/XLA if no CUDA
has_tpu = False
if not has_cuda:
    r2 = subprocess.run(
        [sys.executable, "-c",
         "import torch_xla.core.xla_model as xm; print(xm.xla_device())"],
        capture_output=True, text=True
    )
    has_tpu = r2.returncode == 0 and bool(r2.stdout.strip())
    if has_tpu:
        print(f'TPU device: {r2.stdout.strip()}')

if has_cuda:
    get_ipython().system('nvidia-smi')
    import torch
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    DEVICE = "cuda"
    print(f'\n✓ GPU: {name} ({vram:.1f} GB)')
    if vram < 35:
        print('⚠️  <40 GB VRAM — use preset="large" (100M) instead of "xl"')
    else:
        print('✓ Ready for xl model')
elif has_tpu:
    DEVICE = "xla"
    print(f'\n✓ TPU detected — DEVICE={DEVICE!r}')
    print('  Tip: v2/v3 TPUs prefer preset="large"; v4/v5 handle "xl"')
else:
    DEVICE = "cpu"
    print('\n' + '='*60)
    print('❌ No GPU or TPU found!')
    print('FIX: Runtime → Disconnect and delete runtime')
    print('     Runtime → Change runtime type → A100 / T4 / TPU')
    print('     Then re-run all cells from the top.')
    print('='*60)

print(f'\nDEVICE = {DEVICE!r}')


In [ ]:
# 2) Clone repo with PAT for push access
# ─────────────────────────────────────────────────────────────────────────────
# Token: add a Colab Secret named  GITHUB  (Settings → Secrets → Add new secret)
# The secret is read fresh every time this cell runs, so you can add it mid-session.
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess

_token = ""
try:
    from google.colab import userdata as _ud
    _token = (_ud.get("GITHUB") or "").strip()
except Exception:
    pass

if _token:
    # Export to env so train.py subprocesses can read it
    os.environ["GITHUB"] = _token
    os.environ["GITHUB_TOKEN"] = _token  # legacy compat
    # Write to ~/.git-credentials so git always authenticates, even in subprocesses
    _creds = "https://" + _token + "@github.com\n"
    with open(os.path.expanduser("~/.git-credentials"), "w") as _f:
        _f.write(_creds)
    subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
    print("Token ready (stored in git credential store)")
else:
    print("No GITHUB secret found — checkpoints will not persist to Git")
    print("  Add it: Settings (key icon) → Secrets → + Add new secret → name: GITHUB")

_base = "https://github.com/269652/neuroslm.git"
REPO  = f"https://{_token}@github.com/269652/neuroslm.git" if _token else _base

if os.path.exists("/content/neuroslm/.git"):
    print("Repo already present - updating remote and pulling latest...")
    subprocess.run(["git", "remote", "set-url", "origin", REPO],
                   cwd="/content/neuroslm", check=False)
    subprocess.run(["git", "pull", "--ff-only"],
                   cwd="/content/neuroslm", check=False)
    os.chdir("/content/neuroslm")
else:
    os.chdir("/content")
    get_ipython().system(f"git clone {REPO}")
    os.chdir("/content/neuroslm")


# Switch to tpu branch (has --overwrite_ckpt and latest features)
subprocess.run(["git", "fetch", "origin"], check=False)
subprocess.run(["git", "checkout", "tpu"], check=False)
subprocess.run(["git", "pull", "origin", "tpu", "--ff-only"], check=False)

subprocess.run(["git", "config", "user.email", "train@neuroslm"], check=False)
subprocess.run(["git", "config", "user.name",  "NeuroSLM Train"],  check=False)
print(f"Working directory: {os.getcwd()}")


In [ ]:
# 3) Install deps + Git LFS + restore checkpoints + DNA
import torch, os

# Accept CUDA or TPU; only block on plain CPU
_has_cuda = torch.cuda.is_available()
_has_tpu  = False
if not _has_cuda:
    try:
        import torch_xla.core.xla_model as xm
        xm.xla_device()
        _has_tpu = True
    except Exception:
        pass

if not _has_cuda and not _has_tpu:
    raise RuntimeError(
        "No GPU or TPU! Go to Runtime → Disconnect and delete runtime → "
        "Change runtime type → A100/T4 or TPU → re-run from cell 1"
    )

if _has_cuda:
    print(f"✓ torch {torch.__version__}  CUDA GPU={torch.cuda.get_device_name(0)}")
else:
    print(f"✓ torch {torch.__version__}  TPU/XLA backend")

# Install deps (torch is already from Colab — don't touch it)
get_ipython().system('pip install -q numpy tiktoken datasets tqdm einops networkx transformers')
get_ipython().system('pip install -q --no-deps accelerate')
if _has_tpu:
    get_ipython().system('pip install -q cloud-tpu-client')

# Git LFS
get_ipython().system('apt-get install -y git-lfs -qq 2>/dev/null')
get_ipython().system('git lfs install')
get_ipython().system('git lfs pull')

# Restore LFS checkpoints + memory + evolved DNA
import glob, shutil
os.makedirs('/content/checkpoints', exist_ok=True)
for ext in ['*.pt', '*.mem', '*.mem.json', '*.dna.json']:
    for c in sorted(glob.glob(f'lfs_checkpoints/{ext}')):
        dest = os.path.join('/content/checkpoints', os.path.basename(c))
        if not os.path.exists(dest):
            shutil.copy2(c, dest)
            print(f"  Restored: {os.path.basename(c)}")

CKPT_DIR = locals().get('CKPT_DIR', '/content/checkpoints')
existing = sorted(glob.glob(CKPT_DIR + '/*.pt'))
dna_files = sorted(glob.glob('checkpoints/*.dna.json'))
if existing:
    print(f"\n✓ {len(existing)} checkpoint(s). Latest: {existing[-1]}")
    ckpt = torch.load(existing[-1], map_location='cpu', weights_only=False)
    print(f"  Step {ckpt.get('step', '?')}")
    has_dna = 'module_genomes' in ckpt or 'epigenetic' in ckpt
    print(f"  Evolved DNA in checkpoint: {has_dna}")
    if dna_files:
        print(f"  DNA snapshots: {len(dna_files)} (.dna.json files)")
    del ckpt
else:
    print("No checkpoints — training from scratch")


In [ ]:
# 4) Configuration — change these to control training
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PRESET     = "xl"        # "large" (100M, T4/TPU v2-3) | "xl" (350M, A100/TPU v4+)
TOTAL_STEPS = 20_000
BATCH_SIZE  = 1
GRAD_ACCUM  = 4
SAVE_EVERY  = 500
CKPT_DIR    = '/content/checkpoints'
MODE        = "mix"
CHAT_RATIO  = 0.6
OVERWRITE_CKPT = True   # True = overwrite existing checkpoint; False = keep step-numbered files

import torch

# DEVICE was set in cell 1; fall back gracefully if cells run out of order
if 'DEVICE' not in dir():
    if torch.cuda.is_available():
        DEVICE = "cuda"
    else:
        try:
            import torch_xla.core.xla_model as xm; xm.xla_device(); DEVICE = "xla"
        except Exception:
            DEVICE = "cpu"

if DEVICE == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 35 and PRESET == "xl":
        print(f"⚠️ Only {vram:.0f} GB VRAM — falling back to large preset (100M)")
        PRESET     = "large"
        BATCH_SIZE = 1
    else:
        print(f"✓ {vram:.0f} GB VRAM — using {PRESET} preset (batch={BATCH_SIZE})")
elif DEVICE == "xla":
    print(f"✓ TPU runtime — using {PRESET} preset (batch={BATCH_SIZE})")
    print("  Tip: v2/v3 HBM is 8 GB/core; set PRESET=\"large\" if OOM")

print(f"\nTraining plan:")
print(f"  Preset:     {PRESET}")
print(f"  Device:     {DEVICE}")
print(f"  Steps:      {TOTAL_STEPS:,}")
print(f"  Batch:      {BATCH_SIZE} x {GRAD_ACCUM} grad_accum = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"  Mode:       {MODE} (chat_ratio={CHAT_RATIO})")
print(f"  Save every: {SAVE_EVERY}")


In [ ]:
# 5) ABLATION: Baseline vs Full Model (1000 steps each)
# ╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋
# Trains both models for 1000 steps to confirm bio modules improve over a
# param-matched vanilla transformer BEFORE committing to the full 100K run.
import os, subprocess

# Refresh credentials from Colab secret (safe to re-run; no-op if already set)
_tok = ""
try:
    from google.colab import userdata as _ud
    _tok = (_ud.get("GITHUB") or "").strip()
except Exception:
    _tok = os.environ.get("GITHUB", "").strip()
if _tok:
    os.environ["GITHUB"] = _tok
    os.environ["GITHUB_TOKEN"] = _tok
    with open(os.path.expanduser("~/.git-credentials"), "w") as _cf:
        _cf.write("https://" + _tok + "@github.com\n")
    subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
    print("Credentials refreshed (" + str(len(_tok)) + " chars)")
else:
    print("No GITHUB token -- checkpoint push will be skipped")

os.environ["PYTHONUNBUFFERED"] = "1"

print("=" * 60)
print("  ABLATION: Baseline vs Full Model (1000 steps each)")
print("=" * 60)

# --- Baseline: param-matched vanilla transformer ---
print("\n▶ Training BASELINE (param-matched vanilla transformer)...")
_bl_cmd = (f"cd /content/neuroslm && python -u -m neuroslm.train"
           f" --preset {PRESET}"
           f" --steps 1000"
           f" --batch_size {BATCH_SIZE}"
           f" --grad_accum {GRAD_ACCUM}"
           f" --ckpt_dir /content/checkpoints_baseline"
           f" --device {DEVICE}"
           f" --mode {MODE}"
           f" --chat_ratio {CHAT_RATIO}"
           f" --save_every 1000"
           f" --log_every 50"
           f" --seed 42"
           f" --baseline"
           + (" --overwrite_ckpt" if OVERWRITE_CKPT else ""))
print(_bl_cmd)
!{_bl_cmd}

# --- Full model: all bio modules ---
print("\n▶ Training FULL MODEL (bio modules)...")
_fm_cmd = (f"cd /content/neuroslm && python -u -m neuroslm.train"
           f" --preset {PRESET}"
           f" --steps 1000"
           f" --batch_size {BATCH_SIZE}"
           f" --grad_accum {GRAD_ACCUM}"
           f" --ckpt_dir /content/checkpoints_full"
           f" --device {DEVICE}"
           f" --mode {MODE}"
           f" --chat_ratio {CHAT_RATIO}"
           f" --save_every 1000"
           f" --log_every 50"
           f" --seed 42"
           + (" --overwrite_ckpt" if OVERWRITE_CKPT else ""))
print(_fm_cmd)
!{_fm_cmd}

print("\n✓ Both ablation runs complete. Run next cell to compare.")

In [ ]:
# 5b) Compare ablation results
import glob, torch

print("=" * 60)
print("  ABLATION RESULTS: 1000 steps")
print("=" * 60)

for label, ckpt_dir in [
    ("BASELINE (param-matched vanilla)", "/content/checkpoints_baseline"),
    ("FULL MODEL  (bio modules)",        "/content/checkpoints_full"),
]:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/*.pt'))
    if ckpts:
        ckpt = torch.load(ckpts[-1], map_location='cpu', weights_only=False)
        step = ckpt.get('step', '?')
        cfg_d = ckpt.get('cfg', {})
        n_params = sum(v.numel() for v in ckpt['model'].values())
        is_baseline = cfg_d.get('baseline', False)
        print(f"\n  {label}:")
        print(f"    Params:     {n_params/1e6:.2f}M")
        print(f"    Step:       {step}")
        print(f"    Baseline:   {is_baseline}")
        del ckpt
    else:
        print(f"\n  {label}: No checkpoint found in {ckpt_dir}/")

print("\n" + "=" * 60)
print("  Check step-by-step loss in the training output above.")
print("  FULL MODEL loss < BASELINE → bio modules help → run cell 6.")
print("  FULL MODEL loss ≥ BASELINE → tune loss weights first.")
print("=" * 60)

In [ ]:
# 6) Full training — RESUMES AUTOMATICALLY from latest checkpoint
# Run this after the ablation (cells 5 + 5b) confirms bio modules help.
import os, sys, subprocess

# Refresh credentials from Colab secret (safe to re-run; no-op if already set)
_tok = ""
try:
    from google.colab import userdata as _ud
    _tok = (_ud.get("GITHUB") or "").strip()
except Exception:
    _tok = os.environ.get("GITHUB", "").strip()
if _tok:
    os.environ["GITHUB"] = _tok
    os.environ["GITHUB_TOKEN"] = _tok
    with open(os.path.expanduser("~/.git-credentials"), "w") as _cf:
        _cf.write("https://" + _tok + "@github.com\n")
    subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
    print("Credentials refreshed (" + str(len(_tok)) + " chars)")
else:
    print("No GITHUB token -- checkpoint push will be skipped")

os.environ["PYTHONUNBUFFERED"] = "1"

_cmd = (f"cd /content/neuroslm && python -u -m neuroslm.train"
        f" --preset {PRESET}"
        f" --steps {TOTAL_STEPS}"
        f" --batch_size {BATCH_SIZE}"
        f" --grad_accum {GRAD_ACCUM}"
        f" --ckpt_dir {CKPT_DIR}"
        f" --device {DEVICE}"
        f" --meta"
        f" --mode {MODE}"
        f" --chat_ratio {CHAT_RATIO}"
        f" --save_every {SAVE_EVERY}"
        f" --resume latest"
        + (" --overwrite_ckpt" if OVERWRITE_CKPT else ""))
print(_cmd)
!{_cmd}

In [ ]:
# 7) Upload checkpoint + memory + DNA to GitHub
# Reads the GITHUB secret fresh in case env var was lost.
import glob, shutil, os, subprocess

CKPT_DIR = locals().get("CKPT_DIR", "/content/checkpoints")

# Re-read token (works even if cell 2 was not run this session)
_token = os.environ.get("GITHUB", "").strip()
if not _token:
    try:
        from google.colab import userdata as _ud
        _token = (_ud.get("GITHUB") or "").strip()
        if _token:
            os.environ["GITHUB"] = _token
            os.environ["GITHUB_TOKEN"] = _token
            _creds = "https://" + _token + "@github.com\n"
            with open(os.path.expanduser("~/.git-credentials"), "w") as _f:
                _f.write(_creds)
            subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
    except Exception:
        pass

ckpts = sorted(glob.glob(CKPT_DIR + "/*.pt"))
mems  = sorted(glob.glob(CKPT_DIR + "/*.mem"))
dnas  = sorted(glob.glob(CKPT_DIR + "/*.dna.json"))
files = ckpts[-2:] + mems[-2:] + dnas[-2:]

if not files:
    print("No checkpoints in " + CKPT_DIR + ". Train first (cell 6).")
else:
    if _token:
        remote = "https://" + _token + "@github.com/269652/neuroslm.git"
        subprocess.run(["git", "remote", "set-url", "origin", remote], check=False)
    else:
        print("Warning: no GITHUB token -- push may fail")

    os.makedirs("lfs_checkpoints", exist_ok=True)
    staged = []
    for f in files:
        dest = os.path.join("lfs_checkpoints", os.path.basename(f))
        shutil.copy2(f, dest)
        staged.append(os.path.basename(f))
    print("Staging: " + str(staged))

    subprocess.run(["git", "add", "-f", "lfs_checkpoints/"], check=False)
    r_c = subprocess.run(
        ["git", "commit", "--allow-empty", "-m", "chkpt: " + staged[-1]],
        capture_output=True, text=True)
    print(r_c.stdout.strip() or r_c.stderr.strip())

    r_p = subprocess.run(["git", "push"], capture_output=True, text=True)
    if r_p.returncode == 0:
        print("Pushed: " + str(staged))
    else:
        print("Push failed:\n" + r_p.stderr)
        print("If you see denied: your PAT needs repo scope.")
        print("New token: github.com/settings/tokens -> Generate classic token -> check repo")

In [ ]:
# 7) Benchmarks — HellaSwag, ARC-Easy, ARC-Challenge, MMLU
# Compares against SmolLM2, TinyLlama, Phi-3, Qwen2.5, Llama-3.1
import glob
ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f"Benchmarking: {latest}")
    !python -m neuroslm.benchmarks --ckpt {latest} --preset {PRESET} --device {DEVICE} --max_samples 500
else:
    print("No checkpoints. Train first.")

In [ ]:
# 8) Generate text
import glob
ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f'Using: {latest}')
    !python -m neuroslm.generate --ckpt {latest} --preset {PRESET} --prompt "Explain the theory of relativity in simple terms" --max_new 512 --temperature 0.7 --top_k 40 --device {DEVICE}
else:
    print('No checkpoints. Train first.')

In [ ]:
# 9) Intelligence Metrics — consciousness, identity drift, causal density
import glob, torch, sys
sys.path.insert(0, '.')
from neuroslm.config import PRESETS
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
assert ckpts, "No checkpoints. Train first."
latest = ckpts[-1]

tok = Tokenizer()
cfg = PRESETS[PRESET]()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to(DEVICE)
ckpt = torch.load(latest, map_location="cuda", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)
brain.eval()
n_params = sum(p.numel() for p in brain.parameters())
print(f"Model: {n_params/1e6:.1f}M params, step {ckpt.get('step', '?')}")

# Load memory if available
import os
mem_path = latest.replace('.pt', '.mem')
if os.path.exists(mem_path):
    brain.load_memory_checkpoint(mem_path)
    print(f"Memory loaded: {mem_path}")

# Show intelligence metrics
brain.metrics.observe_narrative(brain.narrative_system)
brain.metrics.observe_memory(brain.episodic, brain.consolidated, brain.causal)
snap = brain.metrics.snapshot()
print("\n" + "=" * 60)
print("  INTELLIGENCE & CONSCIOUSNESS METRICS")
print("=" * 60)
for k, v in snap.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")
print("=" * 60)
print(f"\n  Causal rules learned: {brain.causal.stats()}")
print(f"  Memory gate stats: {brain.comprehension_gate.stats()}")

In [ ]:
# 10) Inspect Module Algorithms — Genome alleles → Opcodes → Lisp → Module Params
# The genome IS the program. Alleles directly encode opcode logits + operands.
# Pipeline: ModuleGenome.alleles → decompile → Lisp → extract params → push to modules
import glob, torch, sys, os
sys.path.insert(0, '.')
from neuroslm.config import PRESETS
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
assert ckpts, "No checkpoints. Train first."
latest = ckpts[-1]

tok = Tokenizer()
cfg = PRESETS[PRESET]()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to("cpu")
ckpt = torch.load(latest, map_location="cpu", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)

# Restore module genomes if saved
if 'module_genomes' in ckpt and hasattr(brain, 'module_genomes'):
    from neuroslm.dna.compiler import ModuleGenomePool
    brain.module_genomes = ModuleGenomePool.from_state(ckpt['module_genomes'])
    brain._recompile_all_genomes()
    print(f"✓ Restored module genomes from step {ckpt.get('step', '?')}")

brain.eval()
print(f"Loaded step {ckpt.get('step', '?')}\n")

# Show compilation report
if hasattr(brain, 'genome_compiler'):
    print(brain.genome_compilation_report())

    # Show genome summaries — what each module's algorithm looks like
    print("\n" + "=" * 60)
    print("  MODULE GENOME ALGORITHMS (genome IS the program)")
    print("=" * 60)
    for region, genome in brain.module_genomes.active_all().items():
        print(f"\n  {genome.summary()}")
        print(f"    Generation: {genome.generation}, Fitness: {genome.fitness:.4f}")
        # Show extracted params that actually control the module
        env = brain.genome_compiler.get_env(region)
        param_keys = [k for k in env if not k.startswith('_') and k not in
                      ('__region__', 'layers', 'connections', 'learning_rule',
                       'projections', 'nt_production')]
        if param_keys:
            params_str = ", ".join(f"{k}={env[k]:.3f}" for k in sorted(param_keys)
                                   if isinstance(env.get(k), (int, float)))
            print(f"    Params → module: {params_str}")

    print("\n" + "=" * 60)
    print("  COMPILED LISP (decompiled from genome alleles)")
    print("=" * 60)
    for name, lisp_src in brain.get_all_module_lisp().items():
        print(f"\n{'─' * 60}")
        print(lisp_src)

    # Save to files
    out_dir = brain.genome_compiler.save_all_lisp('compiled_programs')
    print(f"\n✓ Saved compiled .lisp files to {out_dir}/")
    for f in sorted(os.listdir(out_dir)):
        print(f"  {f}")
else:
    print("No genome compiler found (baseline model)")

## Multi-day training workflow

1. **First run:** Run cells 1–6 sequentially. Training starts from scratch.
2. **Runtime disconnects:** Colab disconnects after ~12h. Checkpoints are auto-saved + pushed.
3. **Resume:** Re-run cells 1–5. Cell 5 auto-detects the latest checkpoint and continues.
4. **Track progress:** Run cell 7 (benchmarks) periodically to see improvement vs SOTA.
5. **Ablation:** Run cell 6b instead of 5 to compare baseline vs full model (critical first step).
6. **Target:** 100K steps ≈ 800M tokens at batch=4, ctx=2048.

### Training data pipeline
- **Text:** FineWeb-Edu (10B tokens curated web), Cosmopedia, TinyStories
- **Chat:** OpenHermes-2.5, UltraChat-200k, WildChat-1M, SlimOrca, hh-rlhf, Dolly
- **Mode `mix`:** 60% chat + 40% text (tunable via `CHAT_RATIO`)

### What gets persisted across restarts
- `.pt` checkpoint (model weights + optimizer + genome state + epigenetic marks)
- `.mem` memory checkpoint (episodic + consolidated + causal)
- `.dna.json` evolved DNA snapshot (inspectable JSON)

### Benchmark targets (258M xl preset)
| Benchmark | Random | Target | SOTA ~300M (SmolLM2) |
|-----------|--------|--------|---------------------|
| HellaSwag | 0.25 | >0.35 | 0.42 |
| ARC-Easy | 0.25 | >0.40 | 0.48 |
| ARC-Challenge | 0.25 | >0.28 | 0.30 |
| MMLU | 0.25 | >0.28 | 0.30 |